# Nivel 3 — Metadatos y coste
Extraer tokens y modelo de la respuesta y estimar el coste de la llamada.

In [1]:
import os

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()  # lee OPENROUTER_API_KEY de practicas/.env

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

MODELO = "openai/gpt-4o-mini"
SYSTEM = "Eres un experto en estimación de proyectos de software con 15 años de experiencia."
MENSAJE = "¿Cuánto se tarda en hacer una app de login?"

r = client.chat.completions.create(
    model=MODELO,
    messages=[
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content": MENSAJE},
    ],
)
print(r.choices[0].message.content)

La duración para desarrollar una aplicación de inicio de sesión (login) puede variar en función de varios factores, incluyendo la complejidad del proyecto, el entorno tecnológico y los requisitos específicos. Sin embargo, aquí hay una estimación general:

1. **Requisitos y diseño**: 1-2 semanas
   - Reuniones con el cliente para definir requisitos.
   - Diseño de la interfaz de usuario (UI) y experiencia de usuario (UX).

2. **Desarrollo del backend**: 1-3 semanas
   - Configuración del servidor y base de datos.
   - Implementación de la lógica para gestionar cuentas de usuario (registro, autenticación y autorización).

3. **Desarrollo del frontend**: 1-2 semanas
   - Creación de la interfaz del usuario para el login.
   - Implementación de la integración con el backend.

4. **Pruebas**: 1-2 semanas
   - Pruebas unitarias.
   - Pruebas funcionales y de usabilidad.
   - Corrección de errores.

5. **Despliegue y documentación**: 1 semana
   - Preparación para el lanzamiento.
   - Documen

## Metadatos

In [2]:
tokens_entrada = r.usage.prompt_tokens
tokens_salida = r.usage.completion_tokens

print("Modelo utilizado:", r.model)
print("Tokens de entrada:", tokens_entrada)
print("Tokens de salida: ", tokens_salida)
print("Tokens totales:   ", r.usage.total_tokens)

Modelo utilizado: openai/gpt-4o-mini
Tokens de entrada: 40
Tokens de salida:  324
Tokens totales:    364


## Coste estimado
Precios en USD por **1M de tokens**, sacados de https://platform.openai.com/pricing (gpt-4o-mini).

In [3]:
PRECIO_ENTRADA_1M = 0.15  # USD / 1M tokens de entrada
PRECIO_SALIDA_1M = 0.60   # USD / 1M tokens de salida

coste_entrada = tokens_entrada * PRECIO_ENTRADA_1M / 1_000_000
coste_salida = tokens_salida * PRECIO_SALIDA_1M / 1_000_000
coste_total = coste_entrada + coste_salida

print(f"Coste entrada: ${coste_entrada:.8f}")
print(f"Coste salida:  ${coste_salida:.8f}")
print(f"Coste total:   ${coste_total:.8f}")
print(f"Llamadas así por 1 USD: ~{1 / coste_total:,.0f}")

Coste entrada: $0.00000600
Coste salida:  $0.00019440
Coste total:   $0.00020040
Llamadas así por 1 USD: ~4,990


## Comparación con el coste real
OpenRouter devuelve lo que te ha cobrado en `usage.cost`.

In [4]:
coste_real = getattr(r.usage, "cost", None)
print(f"Estimado: ${coste_total:.8f}")
print(f"Real:     ${coste_real:.8f}" if coste_real is not None else "OpenRouter no devolvió coste")

Estimado: $0.00020040
Real:     $0.00020040


## Observaciones
- La salida cuesta 4× más que la entrada: las respuestas largas (como la del rol experto) encarecen la llamada.
- El system prompt también cuenta como tokens de entrada, y se paga en **cada** llamada.